<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-09c-vector-database-filtered-search-qdrant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Discussion 9 (continued) - Vector databases and filtered search: Qdrant, Pinecone and the post-filter trap

Chapter 8 built retrieval on FAISS and Chroma. Production systems usually need more than "the 10 nearest vectors": they need *the 10 nearest vectors **that also satisfy a condition*** - this customer's documents, this product line, this date range. How a vector database handles that condition decides whether you get the right answers. This lab measures the problem on real text and shows how to use a vector database API (Qdrant's) for it.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Runs on the free Colab CPU in about five minutes (it downloads a small embedding model and a text corpus).

*Original code written for this course. Data: the 20 Newsgroups text collection (via scikit-learn), embedded with the open `all-MiniLM-L6-v2` model. Qdrant (Apache-2.0) is used through its Python client's local mode; FAISS (MIT) provides the approximate index. Pinecone is a proprietary managed service and is described from its public documentation, not run.*

In [ ]:
%pip install -q qdrant-client faiss-cpu sentence-transformers scikit-learn numpy

In [ ]:
import time, numpy as np, faiss
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer
rng = np.random.default_rng(0)

REMOVE = ("headers", "footers", "quotes")
train = fetch_20newsgroups(subset="train", remove=REMOVE); test = fetch_20newsgroups(subset="test", remove=REMOVE)
def clean(data, n, seed):
    idx = [i for i, t in enumerate(data.data) if len(t.strip()) > 200]
    idx = np.random.default_rng(seed).permutation(idx)[:n]
    return [data.data[i].strip()[:600] for i in idx], [data.target_names[data.target[i]] for i in idx]
docs, cats = clean(train, 6000, 0)
queries, qcats = clean(test, 150, 1)
GROUP = {c: c.split(".")[0] for c in train.target_names}                  # comp, rec, sci, talk, alt, misc, soc
groups = [GROUP[c] for c in cats]
print(len(docs), "documents,", len(set(cats)), "categories,", len(set(groups)), "groups;", len(queries), "queries")
print("documents per group:", {g: groups.count(g) for g in sorted(set(groups))})

In [ ]:
t0 = time.time()
enc = SentenceTransformer("all-MiniLM-L6-v2")                              # 384 dimensions, CPU-friendly
V = enc.encode(docs, batch_size=64, convert_to_numpy=True, normalize_embeddings=True).astype("float32")
Q = enc.encode(queries, batch_size=64, convert_to_numpy=True, normalize_embeddings=True).astype("float32")
print(f"embedded {len(V)} documents and {len(Q)} queries in {time.time()-t0:.0f}s -> {V.shape[1]}-dimensional unit vectors")

## Part 1 - The task: nearest neighbours *within a filter*

We ask: for each query, which 10 documents are most similar **among those in a chosen group**? The exact answer is easy to compute with a matrix product on a mask; it is our ground truth. Two filters: a broad one (all `sci` documents) and a selective one (only `sci.space`).

In [ ]:
cats_arr = np.array(cats); groups_arr = np.array(groups)
FILTERS = {"broad: group = sci": groups_arr == "sci", "selective: category = sci.space": cats_arr == "sci.space"}
for k, m in FILTERS.items(): print(f"{k:34} matches {m.sum():5} of {len(m)} documents ({m.mean():.1%})")

def exact_filtered(mask, k=10):
    sims = Q @ V.T; sims[:, ~mask] = -9
    return np.argsort(-sims, axis=1)[:, :k]
TRUTH = {name: exact_filtered(mask) for name, mask in FILTERS.items()}

## Part 2 - The post-filter trap

The simplest way to add a filter to an ANN index that has none is to **search first and filter afterwards**: ask for the top *k'* neighbours, drop those that fail the condition, keep what is left. It is cheap and wrong when the filter is selective, because the nearest neighbours of the query are mostly *not* in the filtered set. We measure it on a FAISS HNSW index, overfetching 10, 50, 200 and 1000 candidates.

In [ ]:
index = faiss.IndexHNSWFlat(V.shape[1], 32, faiss.METRIC_INNER_PRODUCT); index.hnsw.efConstruction = 200; index.add(V); index.hnsw.efSearch = 256

def post_filter(mask, overfetch, k=10):
    _, I = index.search(Q, overfetch)
    out, short = [], 0
    for row in I:
        keep = [j for j in row if j >= 0 and mask[j]][:k]; short += len(keep) < k; out.append(keep)
    return out, short

def recall(found, truth):
    return float(np.mean([len(set(f) & set(t)) / len(t) for f, t in zip(found, truth)]))

print(f"{'filter':34}{'fetched':>9}{'recall@10':>11}{'queries with <10 results':>27}")
for name, mask in FILTERS.items():
    for over in (10, 50, 200, 1000):
        found, short = post_filter(mask, over)
        print(f"{name:34}{over:>9}{recall(found, TRUTH[name]):>11.3f}{short:>20} of {len(Q)}")

**Read it.** Fetching only the 10 nearest and filtering afterwards is badly wrong even for a broad filter (22% of the documents): most queries come back with fewer than 10 results and recall@10 is about 0.2. Recall reaches 0.98 only after fetching **1,000** candidates, a hundred times what you wanted. With the selective filter (5%) it is worse: at 200 candidates recall is about 0.36 and most queries still return fewer than 10 results, and it takes 1,000 candidates to reach about 0.94. And on this small 6,000-document corpus 1,000 candidates is already one sixth of the data; on a corpus of millions the overfetch you would need grows with it. You cannot know the right overfetch in advance, and you pay for a large search to throw most of it away. A database that supports **filtered search natively** avoids the trap.

## Part 3 - Native filtered search with a vector database (Qdrant)

Qdrant stores each vector together with a **payload** (a JSON object of metadata) and applies a payload condition *inside* the search, so the result is the nearest neighbours that satisfy the filter. (Its server builds a filter-aware HNSW graph and uses payload indexes to choose a query strategy; the Python client's **local mode**, used here, runs in-process for development and tests and searches exactly. Production uses the server or Qdrant Cloud.) The API is the same in both.

In [ ]:
from qdrant_client import QdrantClient, models
client = QdrantClient(":memory:")
client.create_collection("news", vectors_config=models.VectorParams(size=V.shape[1], distance=models.Distance.COSINE))
client.upsert("news", points=[models.PointStruct(id=i, vector=V[i].tolist(), payload={"text": docs[i][:120], "category": cats[i], "group": groups[i]}) for i in range(len(V))])
client.create_payload_index("news", field_name="group", field_schema=models.PayloadSchemaType.KEYWORD)      # index the fields you filter on
client.create_payload_index("news", field_name="category", field_schema=models.PayloadSchemaType.KEYWORD)
print("points stored:", client.count("news").count)

def qdrant_filtered(field, value, k=10):
    flt = models.Filter(must=[models.FieldCondition(key=field, match=models.MatchValue(value=value))])
    out = []
    for q in Q:
        res = client.query_points("news", query=q.tolist(), query_filter=flt, limit=k).points
        out.append([p.id for p in res])
    return out

for name, (field, value) in {"broad: group = sci": ("group", "sci"), "selective: category = sci.space": ("category", "sci.space")}.items():
    t0 = time.time(); found = qdrant_filtered(field, value); dt = time.time() - t0
    short = sum(len(f) < 10 for f in found)
    print(f"{name:34} recall@10 {recall(found, TRUTH[name]):.3f} | queries with <10 results: {short} of {len(Q)} | {dt/len(Q)*1000:.1f} ms per query (local, exact)")

**Read it.** The filter is applied as part of the search, so every query returns a full 10 results that satisfy it, and the answer agrees with the exact ground truth (recall 0.999; we did not investigate the tiny difference, which is probably floating-point rounding). Local mode searches exactly, so this shows the API semantics, not the speed or recall of an approximate production index; measure those on a real server with your own data.

The same payload model gives you the other operations a production system needs:

In [ ]:
flt_space = models.Filter(must=[models.FieldCondition(key="category", match=models.MatchValue(value="sci.space"))])
print("count with filter:", client.count("news", count_filter=flt_space, exact=True).count)

# combine conditions: sci.space OR sci.med, but NOT very short documents flagged in the payload
client.set_payload("news", payload={"short": True}, points=[i for i in range(len(docs)) if len(docs[i]) < 300])
flt_or = models.Filter(should=[models.FieldCondition(key="category", match=models.MatchValue(value=c)) for c in ("sci.space", "sci.med")],
                       must_not=[models.FieldCondition(key="short", match=models.MatchValue(value=True))])
hit = client.query_points("news", query=Q[0].tolist(), query_filter=flt_or, limit=3).points
print("OR filter, excluding short documents:", [(p.payload["category"], round(p.score, 3)) for p in hit])

# delete by filter, e.g. a customer asks for their data to be removed
before = client.count("news").count
client.delete("news", points_selector=models.FilterSelector(filter=flt_space))
print(f"deleted by filter: {before} -> {client.count('news').count} points")

## Part 4 - Choosing a store

| | FAISS | Chroma | Qdrant | Pinecone |
|---|---|---|---|---|
| What it is | A similarity-search *library* (MIT) | Embedded or server vector database (Apache-2.0) | Vector database with server, cloud and local mode (Apache-2.0) | Proprietary, fully managed vector database service |
| Metadata and filters | None built in: you keep ids and filter yourself (the post-filter trap above) | Metadata filters on queries | Rich payload filters (match, range, geo, nested; AND/OR/NOT) with payload indexes and a filter-aware HNSW | Metadata filtering and namespaces |
| You run it | In your process | In your process or a server | Your server, Docker, or Qdrant Cloud | Not self-hostable; you use the service |
| Reach for it when | Research, batch jobs, a fixed corpus with no filters | Prototypes and small apps | Production retrieval with filters, updates and deletes you control | You want no infrastructure and accept a managed, paid service |

Whichever you choose: (1) **index the fields you filter on**; (2) **test with your own filters**, not the average query, because selective filters are where systems differ; (3) check how **updates and deletes** behave (the delete-by-filter above is how you honour a data-removal request); (4) measure recall against an exact baseline as Chapter 8 did; (5) look at **quantization** options (scalar, product or binary) if memory is the constraint, and re-measure recall after turning them on.

## What to hand in

The post-filter table for your own selective filter with the overfetch at which recall becomes acceptable, a screenshot or printout of a filtered Qdrant query on your data, and one paragraph choosing a store for a retrieval system with per-tenant data and deletion requests, justified with the measurements above.